# Session 1 - Messages API, Streaming and SDK Setup

**CCAF (Claude Certified Architect Foundations)** - hands-on companion notebook.

Read the explanation, run the cell, look at the output. Each section maps to a block of the session:

| # | Section | What you will see |
|---|---------|-------------------|
| 0 | Setup | Load the API key, create the client |
| 1 | Messages API | Required fields, response object, content blocks, system prompt, `max_tokens`, `stop_reason`, tools, thinking |
| 2 | SDK setup and multi-turn state | Auth, helper functions, history, token growth, auth errors |
| 3 | Streaming | Non-streaming vs streaming, raw events, `text_stream`, partial tool JSON |
| 4 | Lab | One streaming, multi-turn chat client that pulls it all together |

<br>

> Outputs differ on every run (the model is not deterministic). The **structure** is what matters.

## 0. Setup

1. `pip install anthropic python-dotenv`

The SDK reads `ANTHROPIC_API_KEY` from the environment, so you never write the key in code.

In [ ]:
import os

os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."  # replace with your key.

In [ ]:
%pip install anthropic python-dotenv requests 

In [ ]:
# %pip install anthropic python-dotenv
from anthropic import Anthropic

client = Anthropic()               # picks up ANTHROPIC_API_KEY automatically
MODEL = "claude-haiku-5-5"         # a current model ID; small and cheap for a classroom

print("API key loaded:", bool(os.environ.get("ANTHROPIC_API_KEY")))   # prints True/False, never the key

### Warm-up: does Claude remember earlier requests?

Two **separate** requests. If the API kept a session, the second would know the name.

In [ ]:
r1 = client.messages.create(
    model=MODEL, max_tokens=100,
    messages=[{"role": "user", "content": "My name is Vijay. Remember it."}],
)
print("Request 1 ->", r1.content[0].text)

r2 = client.messages.create(                       # a brand-new request, no history
    model=MODEL, max_tokens=100,
    messages=[{"role": "user", "content": "What is my name?"}],
)
print("\nRequest 2 ->", r2.content[0].text)

**Takeaway:** every request is independent. The API is **stateless**; you keep the message list and resend it (section 3).

## 1. Messages API

### 1.1 The request lifecycle

```
Client (browser/app) --1--> Your server --2--> Anthropic API --3--> model
Client (browser/app) <--5-- Your server <--4-- Anthropic API <------+
```

**Why a server?** The API key is secret. Code that runs on the client can be inspected, so a key placed there can be extracted and used on your account. The key stays on your server.

### 1.2 Required fields

| Field | Purpose |
|-------|---------|
| API key | Authenticates the request (SDK handles the `x-api-key` header) |
| `model` | Which Claude model |
| `messages` | The conversation: alternating `user` / `assistant` messages |
| `max_tokens` | A **safety ceiling** on generated tokens, *not* a target |

Your first call:

In [ ]:
message = client.messages.create(
    model=MODEL,
    max_tokens=1000,
    messages=[{"role": "user", "content": "What is quantum computing? Answer in one sentence."}],
)
print(message.content[0].text)

## 1.3 The response object

`message` is a `Message` object. Inspect the fields:

In [ ]:
print("id           :", message.id)             # starts with msg_
print("type         :", message.type)           # always "message"
print("role         :", message.role)           # always "assistant"
print("model        :", message.model)
print("stop_reason  :", message.stop_reason)
print("stop_sequence:", message.stop_sequence)  # None unless a custom stop string matched
print("usage        :", message.usage.input_tokens, "in /", message.usage.output_tokens, "out")

In [ ]:
# The full raw structure, as the API returns it
print(message.model_dump_json(indent=2))

Note `content` is a **list of blocks**, not a string. `usage` is what you are billed on (input + output tokens).

### 1.4 Content blocks

Never assume `content[0]` is text. Loop over the blocks and check `block.type`.

| Block `type` in a response | Meaning |
|----------------------------|---------|
| `text` | Generated text |
| `tool_use` | Claude wants to call **your** tool (`id`, `name`, `input`) |
| `thinking` | Claude's reasoning (extended thinking enabled) |
| `server_tool_use` + `*_tool_result` | A tool **Anthropic** ran (web search, code execution, ...) |

In a request, a plain string is shorthand for one `text` block. You can also send `image`, `document` and `tool_result` blocks.

In [ ]:
def render(message):
    """Return the visible text, handling every block type safely."""
    parts = []
    for block in message.content:
        if block.type == "text":
            parts.append(block.text)
        elif block.type == "tool_use":
            parts.append(f"[tool_use: {block.name}({block.input})]")
        elif block.type == "thinking":
            parts.append(f"[thinking: {block.thinking[:60]}...]")
        else:
            parts.append(f"[{block.type}]")
    return "\n".join(parts)

print(render(message))

### 1.5 Roles and the system prompt

* Only two roles appear in `messages`: **`user`** and **`assistant`**, and they **alternate**.
* The system prompt is **not a role**. It is the top-level `system` parameter.

One question, asked with and without a system prompt:

In [ ]:
question = [{"role": "user", "content": "How do I solve 5x + 2 = 3 for x?"}]

a = client.messages.create(model=MODEL, max_tokens=500, messages=question)
b = client.messages.create(
    model=MODEL, max_tokens=500, messages=question,
    system="You are a patient math tutor. Do not directly answer. Guide the student step by step.",
)

print("A. No system prompt:\n", a.content[0].text)
print("\nB. With tutor system prompt:\n", b.content[0].text)

The system prompt changes **behaviour**, not the question.

> **Gotcha:** the API rejects `system=None`. Only include the parameter when you have a value:

In [ ]:
def chat(messages, system=None, max_tokens=1000):
    params = {"model": MODEL, "max_tokens": max_tokens, "messages": messages}
    if system:                         # add only when set - never pass system=None
        params["system"] = system
    return client.messages.create(**params).content[0].text

print(chat([{"role": "user", "content": "Name one planet."}]))

### 1.6 `max_tokens` and `stop_reason`

`max_tokens` is a **ceiling**. If it is hit, the reply is **cut off silently** - there is no error, so you must check `stop_reason` yourself.

In [ ]:
def show(message):
    print(message.content[0].text)
    print(f"-> stop_reason: {message.stop_reason} | output_tokens: {message.usage.output_tokens}")

print("Normal (max_tokens=1000):")
show(client.messages.create(model=MODEL, max_tokens=1000, messages=question))

print("\nTruncated (max_tokens=15):")
show(client.messages.create(model=MODEL, max_tokens=15, messages=question))

Spot the difference: the second reply stops mid-sentence, `stop_reason` is `max_tokens`, and `output_tokens` equals the limit.

**Custom stop sequences** end generation at a string you choose. The stop string itself is *not* included in the text.

In [ ]:
m = client.messages.create(
    model=MODEL, max_tokens=300,
    messages=[{"role": "user", "content": "List three required request fields, then write ###."}],
    stop_sequences=["###"],
)
print(repr(m.content[0].text))
print("stop_reason  :", m.stop_reason)       # "stop_sequence"
print("stop_sequence:", m.stop_sequence)     # "###"

| `stop_reason` | Meaning | What your code should do |
|---------------|---------|--------------------------|
| `end_turn` | Finished naturally | Use the response |
| `max_tokens` | Hit your limit - **truncated** | Raise the limit or continue |
| `stop_sequence` | Hit your custom stop string | Use the text up to the stop point |
| `tool_use` | Claude wants to call a tool | Run it, send back a `tool_result` |
| `pause_turn` | A server-tool turn paused | Send the response back to let it continue |
| `refusal` | Declined for safety reasons | Show a fallback; do not retry unchanged |

In [ ]:
NEXT_STEP = {
    "end_turn":      "Use the response.",
    "max_tokens":    "Truncated! Raise max_tokens or continue the generation.",
    "stop_sequence": "Use the text up to the stop point.",
    "tool_use":      "Run the tool and reply with a tool_result.",
    "pause_turn":    "Send the response back as-is so Claude can continue.",
    "refusal":       "Show a fallback message; do not retry unchanged.",
}

def next_step(message):
    return NEXT_STEP.get(message.stop_reason, f"Unknown stop_reason: {message.stop_reason}")

print(next_step(m))

### 1.7 Other request parameters

| Parameter | What it does |
|-----------|--------------|
| `temperature` | Randomness, `0.0`-`1.0` (default `1.0`). Session 2 covers this in depth |
| `top_p` / `top_k` | Alternative sampling controls. Change `temperature` **or** `top_p`, not both |
| `stop_sequences` | Custom strings that stop generation |
| `stream` | `True` returns server-sent events (section 4) |
| `metadata` | `{"user_id": "..."}` opaque ID for abuse tracking. **No personal data** |
| `tools` / `tool_choice` | Tool definitions and how Claude may pick them |
| `thinking` | Extended thinking: `{"type": "enabled", "budget_tokens": N}` |

Some newer models restrict sampling parameters, so this cell is wrapped to fail gracefully.

In [ ]:
try:
    m = client.messages.create(
        model=MODEL, max_tokens=300,
        system="You are a concise API instructor.",
        messages=[{"role": "user", "content": "In one line, what does temperature control?"}],
        temperature=0.3,
        metadata={"user_id": "learner-042"},
    )
    print(m.content[0].text)
except Exception as e:      # e.g. a model that rejects this parameter
    print(type(e).__name__, "->", e)

### 1.8 Tools: the `tool_use` block

When Claude wants to call a function, the response contains **two blocks** (a `text` block and a `tool_use` block) and `stop_reason == "tool_use"`. **Claude never runs your code.** You run the tool and reply with a `tool_result` that references the same `id`.

Here we only look at the response.

In [ ]:
tools = [{
    "name": "get_weather",
    "description": "Get the current weather for a city.",
    "input_schema": {
        "type": "object",
        "properties": {"city": {"type": "string"}},
        "required": ["city"],
    },
}]

m = client.messages.create(
    model=MODEL, max_tokens=500, tools=tools, tool_choice={"type": "auto"},
    messages=[{"role": "user", "content": "What's the weather in Bengaluru?"}],
)

print("stop_reason:", m.stop_reason)
for block in m.content:
    print(" -", block.type, "|", block.model_dump(exclude_none=True))

> **Rule of thumb:** `tool_use` = *you* run the tool. `server_tool_use` = *Anthropic* already ran it and the result is in the same response.

### 1.9 Extended thinking

With `thinking` enabled the response gains a `thinking` block that comes **before** the `text` block. `budget_tokens` must be at least 1024 and less than `max_tokens`. Thinking tokens count toward `output_tokens`. If you continue the conversation, pass thinking blocks back **unmodified**.

Some newer models use a different thinking setting, so this is wrapped too.

In [ ]:
try:
    m = client.messages.create(
        model=MODEL, max_tokens=2000,
        thinking={"type": "enabled", "budget_tokens": 1024},
        messages=[{"role": "user", "content": "Is 391 a prime number?"}],
    )
    for block in m.content:
        print(f"[{block.type}]", getattr(block, "thinking", None) or getattr(block, "text", ""))
    print("output_tokens (includes thinking):", m.usage.output_tokens)
except Exception as e:
    print(type(e).__name__, "->", e)

## 2. SDK setup and multi-turn state

### 2.1 Authentication

* The SDK reads **`ANTHROPIC_API_KEY`** from the environment. Prefer this over `Anthropic(api_key=...)`.
* Keep the key in `.env` (loaded with `python-dotenv`) and **never commit it**.
* Official SDKs exist for Python, TypeScript, JavaScript, Go and Ruby. Plain HTTP works too.
* A leaked key must be **revoked** in the console and replaced.
* A key is shown only once when created, so a lost key must be replaced too.

Two common errors you can reproduce safely (they use a fake key, so your real one is never touched):

In [ ]:
from anthropic import AuthenticationError

# Error 1: a request with a WRONG key -> HTTP 401
bad_client = Anthropic(api_key="sk-ant-this-key-is-fake")
try:
    bad_client.messages.create(model=MODEL, max_tokens=10,
                               messages=[{"role": "user", "content": "hi"}])
except AuthenticationError as e:
    print("401 AuthenticationError:", e.message)

The other classic: **no key at all** raises `TypeError: Could not resolve authentication method...` when the client is created or used. The fix is almost always to call `load_dotenv()` **before** `Anthropic()` and to check the `.env` location.

### 2.2 The API is stateless

Claude **does not store your conversation**. To hold a multi-turn chat you must:

1. Keep a list of all messages in **your** code.
2. Send the **complete history** with every request.

### 2.3 Helper functions

(The tiny `chat` helper from section 2 returns only the text. Here we also manage history.)

In [ ]:
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})

messages = []

add_user_message(messages, "Define quantum computing in one sentence.")
answer = chat(messages)
add_assistant_message(messages, answer)          # do not forget this step!

add_user_message(messages, "Write another sentence.")
final_answer = chat(messages)                    # Claude now has the context

print("Turn 1:", answer)
print("Turn 2:", final_answer)

What the history looks like on the wire:

In [ ]:
print(json.dumps(messages, indent=2))

### 2.4 The same follow-up WITHOUT history

This is the symptom behind exam question Q3.

In [ ]:
print(chat([{"role": "user", "content": "Write another sentence."}]))

Claude has no idea what "another" refers to, so it writes about something unrelated.

### 2.5 Input tokens grow every turn

Each request resends everything, so cost and latency grow with the length of the chat.

In [ ]:
history = []
for turn in ["What is a qubit?", "And superposition?", "Give me an analogy."]:
    add_user_message(history, turn)
    m = client.messages.create(model=MODEL, max_tokens=200, messages=history)
    add_assistant_message(history, m.content[0].text)
    print(f"{len(history)} messages -> {m.usage.input_tokens} input tokens")

| Consequence of statelessness | Why it matters |
|------------------------------|----------------|
| History grows each turn | Input tokens, cost and latency grow with it |
| Your app owns the state | Drop a message and Claude "forgets" it |
| Long chats can hit the context limit | Plan to trim or summarise old turns |
| You can edit history | Handy for testing; a mistake here changes behaviour |


## 3. Streaming

A full reply can take 10-30 seconds. Without streaming the user stares at a spinner. With streaming, text appears as it is generated. It is still **one request**; the server forwards chunks as they arrive.

| | Non-streaming | Streaming |
|---|---------------|-----------|
| Call | `messages.create(...)` | `messages.stream(...)` or `stream=True` |
| Response | One JSON `Message` | A series of events |
| User sees text | After the **whole** reply is done | As each chunk arrives |
| `stop_reason` | In the response body | In the `message_delta` event |
| Best for | Short replies, batch jobs | Chat UIs, long replies |

### 3.1 Feel the difference

Same prompt, both ways. We measure **time to first text** and **total time**.

In [ ]:
prompt = [{"role": "user", "content": "Write a 120-word description of a fictional database."}]

# Non-streaming: nothing to show until the whole reply is done
t0 = time.time()
full = client.messages.create(model=MODEL, max_tokens=500, messages=prompt)
t_total = time.time() - t0
print(f"Non-streaming: first text visible after {t_total:.1f}s (= total)")

# Streaming: first chunk arrives quickly
t0, first = time.time(), None
with client.messages.stream(model=MODEL, max_tokens=500, messages=prompt) as stream:
    for text in stream.text_stream:
        if first is None:
            first = time.time() - t0
total = time.time() - t0
print(f"Streaming    : first text visible after {first:.1f}s, total {total:.1f}s")

The total time is about the same, but with streaming the user starts reading almost immediately.

### 3.2 The raw event stream

With `stream=True` you receive events and parse them yourself.

| Event | Meaning |
|-------|---------|
| `message_start` | A new message begins (empty `content`, `stop_reason` is `None`) |
| `content_block_start` | A new block begins (text, tool use, ...) |
| `content_block_delta` | A chunk of the block. **This carries the text you display** |
| `content_block_stop` | The block is complete |
| `message_delta` | Top-level changes - the final **`stop_reason`** and usage |
| `message_stop` | The message is complete |

The API may also send `ping` and `error` events. Handle them without crashing.

In [ ]:
stream = client.messages.create(
    model=MODEL, max_tokens=100, stream=True,
    messages=[{"role": "user", "content": "Say hello in two words."}],
)

for event in stream:
    if event.type == "content_block_delta":
        print(f"{event.type:22} {event.delta.type}: {event.delta.text!r}")
    elif event.type == "message_delta":
        print(f"{event.type:22} stop_reason={event.delta.stop_reason}, output_tokens={event.usage.output_tokens}")
    else:
        print(event.type)

Order for a simple text reply:

```
message_start
  content_block_start
    content_block_delta   (x many)
  content_block_stop
message_delta             <- stop_reason arrives here
message_stop
```

Delta types: `text_delta` (text), `input_json_delta` (partial tool-call JSON), `thinking_delta` (extended thinking).

### 3.3 The helper: `messages.stream`

Use the helper unless you need individual events. `text_stream` yields just the text, and `get_final_message()` returns the **assembled** message (for saving to a DB or appending to the history).

In [ ]:
with client.messages.stream(
    model=MODEL, max_tokens=150,
    messages=[{"role": "user", "content": "Explain streaming in two sentences."}],
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)           # flush=True or the terminal buffers it

    final = stream.get_final_message()             # inside the with-block

print()
print("stop_reason:", final.stop_reason, "| output_tokens:", final.usage.output_tokens)
print("assembled  :", final.content[0].text[:60] + "...")

### 3.4 Partial tool-call JSON

When Claude calls a tool during a stream, the arguments arrive as **fragments** (`input_json_delta`). Each fragment on its own is **not valid JSON**. Accumulate, then parse after `content_block_stop`.

In [ ]:
fragments, tool_name = [], None
stream = client.messages.create(
    model=MODEL, max_tokens=300, stream=True, tools=tools,
    tool_choice={"type": "tool", "name": "get_weather"},     # force the tool call for the demo
    messages=[{"role": "user", "content": "Weather in Chennai?"}],
)

for event in stream:
    if event.type == "content_block_start" and event.content_block.type == "tool_use":
        tool_name = event.content_block.name
    elif event.type == "content_block_delta" and event.delta.type == "input_json_delta":
        fragments.append(event.delta.partial_json)
        print("fragment:", repr(event.delta.partial_json))
    elif event.type == "content_block_stop" and fragments:
        break

print("\nTool:", tool_name)
print("Joined:", "".join(fragments))
print("Parsed:", json.loads("".join(fragments)))             # safe only now

### 3.5 Partial output pitfalls

* **Do not treat partial text as final.** A stream can end early (error, or a `max_tokens` stop). Read `stop_reason` *after* the stream.
* **Tool-call JSON arrives in fragments.** Parse only after the block stops.
* **Flush output** (`flush=True`) or streaming looks broken in a terminal.

## 4. Lab - a streaming, multi-turn chat client

Everything from sections 1-3 in one function: **system prompt + history + streaming + `stop_reason`**.

| Capability | Where it came from |
|------------|--------------------|
| `messages.stream` + `text_stream` | Section 3 |
| `system` only when set | Section 1.5 |
| `max_tokens` handling | Section 1.6 |
| History kept in sync | Section 2 |

In [ ]:
SYSTEM = "You are a concise technical assistant."     # experiment with this

def stream_reply(messages, system=SYSTEM, max_tokens=1000):
    params = {"model": MODEL, "max_tokens": max_tokens, "messages": messages}
    if system:
        params["system"] = system
    with client.messages.stream(**params) as stream:
        for text in stream.text_stream:
            print(text, end="", flush=True)
        final = stream.get_final_message()

    print(f"\n[stop_reason: {final.stop_reason}]")
    if final.stop_reason == "max_tokens":
        print("(reply was cut off - raise max_tokens)")
    return final.content[0].text


def run_chat(turns, system=SYSTEM, max_tokens=1000, keep_history=True):
    """Scripted chat so it runs in a notebook. keep_history=False shows what breaks."""
    messages = []
    for user_text in turns:
        print(f"\nYou: {user_text}\nClaude: ", end="")
        add_user_message(messages, user_text)
        reply = stream_reply(messages, system, max_tokens)
        if keep_history:
            add_assistant_message(messages, reply)      # keep the history in sync
        else:
            messages.pop()                              # also drop the user turn -> every turn is fresh
    return messages

history = run_chat(["What is a token?", "Give me an example."])

**Check:** the second answer builds on the first (the history works) and `[stop_reason: end_turn]` printed after each reply.

### Experiments

Run each one and compare with the baseline above.

In [ ]:
# 1. Different system prompt
_ = run_chat(["What is a token?"], system="You are a pirate. Answer in two short sentences.")

In [ ]:
# 2. Tiny max_tokens -> truncated, stop_reason == "max_tokens"
_ = run_chat(["Explain how the internet works in detail."], max_tokens=30)

In [ ]:
# 3. History NOT kept -> the follow-up loses context
_ = run_chat(["What is a token?", "Give me an example."], keep_history=False)

In [ ]:
# 4. Token usage grows with history (stretch goal)
msgs = []
for q in ["What is a token?", "Give me an example.", "Why does it matter for cost?"]:
    add_user_message(msgs, q)
    with client.messages.stream(model=MODEL, max_tokens=200, system=SYSTEM, messages=msgs) as s:
        for _ in s.text_stream:
            pass
        fm = s.get_final_message()
    add_assistant_message(msgs, fm.content[0].text)
    print(f"turn {len(msgs)//2}: in={fm.usage.input_tokens} out={fm.usage.output_tokens}")

## 5. Wrap-up

### Cheat sheet

| Topic | Remember |
|-------|----------|
| Request | `model` + `messages` + `max_tokens` required. `system` optional, top-level, never `None` |
| Roles | `user` and `assistant`, alternating. System is a parameter, not a role |
| Response | `content` is a **list of blocks**. Also `stop_reason` and `usage` |
| `stop_reason` | `end_turn` / `max_tokens` (truncated) / `stop_sequence` / `tool_use` / `pause_turn` / `refusal` |
| Streaming events | `message_start` > `content_block_start` > `content_block_delta` x N > `content_block_stop` > `message_delta` (stop_reason) > `message_stop` |
| Streaming code | `client.messages.stream()` + `.text_stream` + `.get_final_message()` |
| State | Stateless. Append every user **and** assistant message and resend the whole history |
| Security | Key in `ANTHROPIC_API_KEY` / `.env`, never in client code or git. Requests go through your server |